# Structural entanglement in sequential unlearning — study v2

Implements the frozen protocol in `docs/protocol_v2.md`. Stages, each gated by the previous one:

1. **Setup** — pinned repository commit and packages (every heavy step runs in a fresh subprocess, so no kernel restart is needed and *Save & Run All* works).
2. **Fine-tune** Qwen2.5-0.5B-Instruct on all fictional facts (main + calibration entities) → one shared base checkpoint. **Knowledge gate:** every split's loss must fall to ≤ 0.3× its pre-fine-tuning value.
3. **Smoke test** — one request per condition and method; projects total GPU time.
4. **Calibration** — learning rate × steps grid on separate calibration entities; the frozen rule in `scripts/09_select_hyperparameters.py` picks settings that remove ≥ 50% of targeted knowledge while every non-target split keeps ≥ 80%.
5. **Main runs** — NPO (primary) and GA (replication) × 10 shuffled request orders × forget / control / retain streams, 16 requests each. Gated by `START_MAIN_EXPERIMENT`.
6. **Analysis** — paired effects, the v2 inversion rule, entity spill-over, layer profile, figures.

Artifacts go to `/kaggle/working/study-v2-artifacts`. Everything is resumable: completed runs are skipped on rerun.

## 1. Setup: clone the pinned commit and install pinned packages

In [ ]:
import os
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/ArsalanKaleem/structurale-ntanglement-unlearning.git"
REPOSITORY_COMMIT = "REPLACE_WITH_V2_COMMIT"
PROJECT_DIR = Path("/kaggle/working/structural-entanglement-unlearning")
ARTIFACT_ROOT = Path("/kaggle/working/study-v2-artifacts")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

if len(REPOSITORY_COMMIT) != 40 or any(c not in "0123456789abcdefABCDEF" for c in REPOSITORY_COMMIT):
    raise ValueError("Set REPOSITORY_COMMIT to the published 40-character commit containing this notebook and lock")
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", REPOSITORY_COMMIT], cwd=PROJECT_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=PROJECT_DIR, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip() == REPOSITORY_COMMIT

subprocess.run(
    ["python", "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-kaggle.lock")],
    check=True,
)
subprocess.run(
    ["python", "-m", "pip", "install", "--no-deps", "-e", str(PROJECT_DIR)],
    check=True,
)
# Kaggle's preinstalled torchvision/torchaudio are built for its own PyTorch, not the
# pinned one, and transformers imports torchvision when present. This project uses
# neither, so remove them to avoid "operator torchvision::nms does not exist".
subprocess.run(
    ["python", "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
    check=True,
)
print("Pinned checkout:", REPOSITORY_COMMIT)
print("Project:", PROJECT_DIR)
print("Artifacts:", ARTIFACT_ROOT)

## 2. Inspect the session's Python, PyTorch, CUDA, GPU, and disk (in a subprocess)

In [ ]:
import json
import sys

# Run in a fresh interpreter: this kernel may still hold Kaggle's preinstalled
# packages in memory, while the pinned versions are now on disk.
ENVIRONMENT_PROBE = r"""
import json, os, platform, shutil, sys
from importlib.metadata import version
from pathlib import Path
import torch, transformers

out_path = Path(sys.argv[1])
print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name}; total memory={props.total_memory / 2**30:.2f} GiB; capability={props.major}.{props.minor}")
    print("Current free/total:", torch.cuda.mem_get_info(index))
print("Working disk free bytes:", shutil.disk_usage("/kaggle/working").free)
try:
    print("Session uptime seconds:", float(Path("/proc/uptime").read_text().split()[0]))
except OSError:
    print("Session uptime: unavailable on this runtime")
print("Kaggle-related environment keys:", sorted(k for k in os.environ if k.startswith("KAGGLE_")))
out_path.write_text(json.dumps({
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "working_disk_free_bytes": shutil.disk_usage("/kaggle/working").free,
    "package_versions": {name: version(name) for name in ("torch", "transformers", "tokenizers", "datasets", "accelerate", "numpy", "pandas", "scikit-learn", "PyYAML", "tqdm", "pytest", "huggingface-hub")},
}, indent=2))
"""
environment_path = ARTIFACT_ROOT / "session_environment.json"
subprocess.run([sys.executable, "-c", ENVIRONMENT_PROBE, str(environment_path)], check=True)
session_environment = json.loads(environment_path.read_text())
if not session_environment["cuda_available"]:
    raise RuntimeError("No CUDA GPU is visible. Enable a Kaggle GPU accelerator before proceeding.")

## 3. Inputs, permissions, and study configuration

Read the [Qwen license](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) and set both review flags to `True`. The datasets are the repository's own synthetic files (MIT).

In [ ]:
import copy
import hashlib
import requests
import yaml

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
MAIN_DATASET = PROJECT_DIR / "data" / "fictional_authors_v1.jsonl"
CALIBRATION_DATASET = PROJECT_DIR / "data" / "fictional_authors_calibration_v1.jsonl"
DATASET_VERSION = "fictional-authors-v1-sha256-5644e12fbc7e"
CALIBRATION_VERSION = "fictional-authors-calibration-v1-sha256-79bcfdb4468c"
MODEL_LICENSE_REVIEWED_FOR_THIS_USE = False
DATASET_RIGHTS_REVIEWED_FOR_THIS_USE = False

# Frozen design (docs/protocol_v2.md)
METHODS = ["npo", "ga"]                       # npo is primary, ga is the replication
SEEDS = [17, 23, 29, 31, 37, 41, 43, 47, 53, 59]  # 10 shuffled request orders
REPLACEMENT_SEEDS = [61, 67]                  # only for technically excluded seeds
CALIBRATION_GRID = [(lr, steps) for lr in (1e-6, 3e-6, 1e-5) for steps in (2, 5)]
CALIBRATION_SEED = 1

model_response = requests.get(
    f"https://huggingface.co/api/models/{MODEL_ID}/revision/{MODEL_REVISION}", timeout=30
)
model_response.raise_for_status()
model_info = model_response.json()
assert model_info.get("sha", "").lower() == MODEL_REVISION.lower(), "Hub returned a different model revision"
print("Declared model license:", (model_info.get("cardData") or {}).get("license"))
for path in (MAIN_DATASET, CALIBRATION_DATASET):
    print(path.name, hashlib.sha256(path.read_bytes()).hexdigest())
if not MODEL_LICENSE_REVIEWED_FOR_THIS_USE or not DATASET_RIGHTS_REVIEWED_FOR_THIS_USE:
    raise PermissionError("Review and confirm model and dataset research-use permissions in this cell")

SHARED_BASE = ARTIFACT_ROOT / "finetuned_base"
FINETUNE_REPORT = ARTIFACT_ROOT / "finetuned_base_report.json"
CONFIG_DIR = ARTIFACT_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = {
    "experiment": {
        "name": "study_v2", "seed": 17, "output_dir": "",
        "checkpoint_retention": "resume_only", "shared_base_dir": str(SHARED_BASE),
    },
    "model": {"name_or_path": MODEL_ID, "revision": MODEL_REVISION, "device": "cuda", "max_length": 256},
    "data": {
        "input_jsonl": str(MAIN_DATASET), "version": DATASET_VERSION,
        **{f"max_{split}_examples": 64 for split in (
            "forget", "forget_update", "retain", "control", "control_update", "retain_update")},
    },
    "finetune": {
        "datasets": [str(MAIN_DATASET), str(CALIBRATION_DATASET)],
        "learning_rate": 2e-5, "epochs": 5, "batch_size": 16, "seed": 0, "max_loss_ratio": 0.3,
    },
    "unlearning": {
        "method": "npo", "learning_rate": None, "max_steps_per_request": None, "npo_beta": 0.1,
        "batch_size": 1, "sequence_length": 16, "gradient_accumulation_steps": 1,
    },
    "evaluation": {
        "primary_structure_metric": "cosine_separation_forget_retain", "prompt_template": "{text}",
        "fixed_layer": -1, "extra_layers": [6, 12, 18], "max_eval_examples_per_split": 64,
    },
}


def write_config(name, method, seed, output_dir, learning_rate=None, steps=None,
                 calibration=False, sequence_length=16):
    config = copy.deepcopy(BASE_CONFIG)
    config["experiment"].update({"seed": seed, "output_dir": str(output_dir)})
    config["unlearning"].update({
        "method": method, "learning_rate": learning_rate, "max_steps_per_request": steps,
        "sequence_length": sequence_length,
    })
    if calibration:
        config["data"].update({"input_jsonl": str(CALIBRATION_DATASET), "version": CALIBRATION_VERSION})
    path = CONFIG_DIR / f"{name}.yaml"
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path


def run_complete(run_dir):
    run_json = run_dir / "run.json"
    return run_json.is_file() and json.loads(run_json.read_text()).get("status") == "complete"


print("Study configuration ready")

## 4. Software tests and dataset validation

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)
for path in (MAIN_DATASET, CALIBRATION_DATASET):
    subprocess.run([sys.executable, "scripts/01_prepare_data.py", "--input", str(path),
                    "--output", str(ARTIFACT_ROOT / "validated-data" / path.stem)],
                   cwd=PROJECT_DIR, check=True)

### Measured subprocess runner (wall time and GPU peak per script)

In [ ]:
import csv
import time

MEASURED_RUNNER = ARTIFACT_ROOT / "run_measured.py"
MEASURED_RUNNER.write_text(r"""
import json, runpy, sys, time
from pathlib import Path
import torch

out_path, script, *script_args = sys.argv[1:]
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
sys.argv = [script, *script_args]
started = time.perf_counter()
runpy.run_path(script, run_name="__main__")
if torch.cuda.is_available():
    torch.cuda.synchronize()
Path(out_path).write_text(json.dumps({
    "inner_elapsed_seconds": time.perf_counter() - started,
    "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0,
}))
""")


def run_script_timed(script, args):
    """Run a project script in a fresh interpreter and record wall time and GPU peak."""
    result_path = ARTIFACT_ROOT / "resource-measurements" / "last_run.json"
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.unlink(missing_ok=True)
    started = time.perf_counter()
    subprocess.run(
        [sys.executable, str(MEASURED_RUNNER), str(result_path), str(PROJECT_DIR / script), *args],
        cwd=PROJECT_DIR,
        check=True,
    )
    inner = json.loads(result_path.read_text())
    measurement = {
        "script": script,
        "args": list(args),
        "elapsed_seconds": time.perf_counter() - started,
        "peak_allocated_bytes": inner["peak_allocated_bytes"],
    }
    measurements_dir = ARTIFACT_ROOT / "resource-measurements"
    json_path = measurements_dir / "measurements.json"
    previous = json.loads(json_path.read_text()) if json_path.exists() else []
    previous.append(measurement)
    json_path.write_text(json.dumps(previous, indent=2))
    with open(measurements_dir / "measurements.csv", "w", newline="") as target:
        writer = csv.DictWriter(target, fieldnames=["script", "args", "elapsed_seconds", "peak_allocated_bytes"])
        writer.writeheader()
        for row in previous:
            writer.writerow({**row, "args": " ".join(row.get("args", []))})
    return measurement

## 5. Fine-tune once → shared base checkpoint (knowledge gate)

All 768 fictional facts (main + calibration) are learned, so later unlearning removes knowledge the model actually has. The run stops here if the knowledge gate fails.

In [ ]:
finetune_config = write_config("finetune", "npo", 0, ARTIFACT_ROOT / "unused")
print(run_script_timed("scripts/01b_finetune.py", ["--config", str(finetune_config)]))
finetune_report = json.loads(FINETUNE_REPORT.read_text())
print(json.dumps(finetune_report["loss_ratio_after_over_before"], indent=2))
if not finetune_report["knowledge_gate_passed"]:
    raise RuntimeError("Knowledge gate failed; see finetuned_base_report.json")
print("KNOWLEDGE GATE PASSED. Save a Kaggle Notebook Version.")

## 6. Smoke test — one request per condition and method, plus a GPU-time projection

In [ ]:
SMOKE_ROOT = ARTIFACT_ROOT / "smoke"
smoke = []
for method in METHODS:
    path = write_config(f"smoke_{method}", method, 17, SMOKE_ROOT / method, 3e-6, 2, sequence_length=1)
    smoke.append(run_script_timed("scripts/02_run_baseline.py", ["--config", str(path)]))
    for condition in ("forget", "control", "retain"):
        smoke.append(run_script_timed(
            "scripts/03_run_sequential.py",
            ["--config", str(path), "--condition", condition, "--order", "original"],
        ))
one_request = [m["elapsed_seconds"] for m in smoke if m["script"] == "scripts/03_run_sequential.py"]
per_run_seconds = sum(one_request) / len(one_request) * 16  # conservative: reloads counted per request
calibration_runs = len(METHODS) * len(CALIBRATION_GRID)
main_runs = len(METHODS) * len(SEEDS) * 3
projection_hours = per_run_seconds * (calibration_runs + main_runs) / 3600
smoke_report = {
    "status": "passed",
    "measurements": smoke,
    "peak_allocated_gpu_bytes": max(m["peak_allocated_bytes"] for m in smoke),
    "calibration_runs": calibration_runs,
    "main_runs": main_runs,
    "projected_gpu_hours_calibration_plus_main": projection_hours,
}
(ARTIFACT_ROOT / "smoke-report.json").write_text(json.dumps(smoke_report, indent=2))
print(json.dumps(smoke_report, indent=2))
print("Projected GPU-hours (protocol ceiling 30):", round(projection_hours, 2))

## 7. Calibration on separate entities → frozen hyperparameter selection

Forget stream, original order, 16 requests, on the calibration entities only. `scripts/09_select_hyperparameters.py` applies the frozen rule. The main experiment is blocked unless every method's status is `selected`.

In [ ]:
CALIBRATION_ROOT = ARTIFACT_ROOT / "calibration"
for method in METHODS:
    for lr, steps in CALIBRATION_GRID:
        run_root = CALIBRATION_ROOT / method / f"lr{lr:g}_steps{steps}"
        if run_complete(run_root / "condition_forget_original"):
            continue
        path = write_config(f"calibration_{method}_lr{lr:g}_steps{steps}", method, CALIBRATION_SEED,
                            run_root, lr, steps, calibration=True)
        print(method, lr, steps, run_script_timed(
            "scripts/03_run_sequential.py",
            ["--config", str(path), "--condition", "forget", "--order", "original"],
        ))
SELECTION_PATH = ARTIFACT_ROOT / "calibration_selection.json"
run_script_timed("scripts/09_select_hyperparameters.py", [
    "--runs", str(CALIBRATION_ROOT), "--finetune-report", str(FINETUNE_REPORT),
    "--dataset-name", CALIBRATION_DATASET.name, "--output", str(SELECTION_PATH),
])
selection = json.loads(SELECTION_PATH.read_text())["by_method"]
for method in METHODS:
    print(method, selection[method]["status"], selection[method]["selected"])

## 8. Main-run gate

Set `START_MAIN_EXPERIMENT = True` only after reviewing the knowledge gate, smoke projection, and calibration selection. If a method's calibration status is not `selected`, stop and record an amendment in `docs/protocol_v2.md` first.

In [ ]:
START_MAIN_EXPERIMENT = False  # Change only after reviewing sections 5-7.
for method in METHODS:
    if selection[method]["status"] != "selected":
        raise RuntimeError(f"{method}: calibration status {selection[method]['status']}; amend the protocol first")
SELECTED = {m: selection[m]["selected"] for m in METHODS}
MAIN_ROOT = ARTIFACT_ROOT / "main"
print("Selected:", {m: (s["learning_rate"], s["max_steps_per_request"]) for m, s in SELECTED.items()})
print("Main experiment enabled" if START_MAIN_EXPERIMENT else "Main experiment disabled")

## 9. Main runs: method × seed × {forget, control, retain}, shuffled order, 16 requests

Completed runs are skipped on rerun. Checkpoints are deleted as runs complete; only the shared fine-tuned base persists. Save a Kaggle Notebook Version after each method.

In [ ]:
if not START_MAIN_EXPERIMENT:
    raise RuntimeError("Main experiment is disabled")
import shutil

for method in METHODS:
    hp = SELECTED[method]
    for seed in SEEDS:
        seed_root = MAIN_ROOT / method / f"seed_{seed}"
        names = ["condition_no_update_original"] + [f"condition_{c}_shuffled" for c in ("forget", "control", "retain")]
        if all(run_complete(seed_root / name) for name in names):
            continue
        free = shutil.disk_usage("/kaggle/working").free
        if free < 6 * 1024**3:
            raise RuntimeError(f"Only {free / 1024**3:.1f} GiB free before {method} seed {seed}")
        path = write_config(f"main_{method}_seed{seed}", method, seed, seed_root,
                            hp["learning_rate"], hp["max_steps_per_request"])
        run_script_timed("scripts/02_run_baseline.py", ["--config", str(path)])
        for condition in ("forget", "control", "retain"):
            print(method, seed, condition, run_script_timed(
                "scripts/03_run_sequential.py",
                ["--config", str(path), "--condition", condition, "--order", "shuffled"],
            ))
    print(f"{method} complete. Save a Kaggle Notebook Version now.")

## 10. Analysis: paired effects, v2 inversion rule, spill-over, layer profile, figures

In [ ]:
if not START_MAIN_EXPERIMENT:
    raise RuntimeError("Main experiment is disabled")
ANALYSIS_ROOT = ARTIFACT_ROOT / "analysis"
METRICS = ["cosine_separation_forget_retain", "cosine_separation"] + [
    f"{base}_L{layer}" for layer in (6, 12, 18)
    for base in ("cosine_separation_forget_retain", "cosine_separation")
]
for method in METHODS:
    out = ANALYSIS_ROOT / method
    for metric in METRICS:
        run_script_timed("scripts/06_analyze.py", [
            "--input", str(MAIN_ROOT / method), "--output", str(out), "--metric", metric,
        ])
    for metric in ("cosine_separation_forget_retain", "cosine_separation"):
        analysis_file = out / ("analysis.json" if metric == "cosine_separation" else f"analysis_{metric}.json")
        run_script_timed("scripts/07_apply_protocol.py", [
            "--runs", str(MAIN_ROOT / method), "--analysis", str(analysis_file),
            "--output", str(out / f"protocol_v2_{metric}.json"), "--protocol", "v2",
            "--finetune-report", str(FINETUNE_REPORT), "--dataset-name", MAIN_DATASET.name,
            "--primary-metric", "cosine_separation_forget_retain",
        ])
        run_script_timed("scripts/08_plot_pilot.py", [
            "--runs", str(MAIN_ROOT / method), "--analysis", str(analysis_file),
            "--protocol", str(out / f"protocol_v2_{metric}.json"),
            "--output", str(out / f"figure_{metric}.png"),
        ])
    run_script_timed("scripts/10_spillover.py", [
        "--input", str(MAIN_ROOT / method), "--dataset", str(MAIN_DATASET),
        "--output", str(out / "spillover.json"),
    ])
    verdict = json.loads((out / "protocol_v2_cosine_separation_forget_retain.json").read_text())
    print(method, "PRIMARY VERDICT:", verdict["verdict"], "| valid steps:", verdict["guard"]["valid_steps"])
print("Download", ARTIFACT_ROOT, "(no model weights except finetuned_base, which you may skip).")

## Recovery

Runs record `status: failed` with the exception in `run.json`. Rerunning any section skips completed work and resumes interrupted runs from their last checkpoint. For a new session, attach the previous version's output, copy `study-v2-artifacts` back to `/kaggle/working`, and rerun from section 1. Do not change configuration, data, or the model for existing output directories; the run fingerprint rejects mismatches.